# Futures symbol scanner

Drop **all** your FirstRateData files (any mix of `.txt` and `.csv`) into one folder, set `DATA_DIR`, and run every cell. It backtests the trendline-walk strategy on each symbol and ranks them by profit factor.

- No renaming needed: it reads `.txt` and `.csv`, and pulls the clean symbol from names like `CL_full_1hour_continuous_ratio_adjusted.txt` or `1782_MGC.csv`.
- Self-contained: the detector is embedded in the next cell.
- `pip install numpy pandas` if needed.

## 1. The detector (run once)

In [ ]:
import csv, numpy as np
from datetime import datetime, timezone

_TIME_FORMATS = ("%Y-%m-%d %H:%M:%S","%Y-%m-%d %H:%M","%Y-%m-%dT%H:%M:%S",
                 "%Y-%m-%dT%H:%M","%m/%d/%Y %H:%M:%S","%m/%d/%Y %H:%M",
                 "%Y-%m-%d","%m/%d/%Y")

def parse_ts(s):
    """Parse a time field into epoch seconds. Accepts epoch seconds, epoch
    milliseconds, or a datetime string (FirstRateData 'YYYY-MM-DD HH:MM:SS',
    US 'MM/DD/YYYY HH:MM', ISO 8601, date only)."""
    s=s.strip()
    try:
        v=float(s)
        if v>1e11: v/=1000.0      # milliseconds -> seconds
        return int(v)
    except ValueError:
        pass
    for fmt in _TIME_FORMATS:
        try: return int(datetime.strptime(s,fmt).replace(tzinfo=timezone.utc).timestamp())
        except ValueError: continue
    raise ValueError("unrecognized time format: %r" % s)

def load_series(path):
    """Load OHLC from a CSV with columns time,open,high,low,close[,volume].
    Time may be epoch (s or ms) OR a datetime string; a header row is skipped
    automatically. Returns T,O,H,L,C arrays, T in epoch seconds, sorted ascending."""
    T=[];O=[];H=[];L=[];C=[]
    with open(path, newline="") as f:
        for r in csv.reader(f):
            if len(r)<5: continue
            try: ts=parse_ts(r[0]); o=float(r[1]); h=float(r[2]); l=float(r[3]); c=float(r[4])
            except (ValueError, IndexError): continue   # header or junk row
            T.append(ts);O.append(o);H.append(h);L.append(l);C.append(c)
    if not T: raise ValueError("no rows parsed from %s" % path)
    idx=np.argsort(T)
    return (np.array(T)[idx],np.array(O,float)[idx],np.array(H,float)[idx],
            np.array(L,float)[idx],np.array(C,float)[idx])

def exit_sim(O,H,L,C,e21,e200,A,t0,entry,stop,d):
    """Replay the exit engine for one position; returns the realized R multiple.
    Mirrors the manager inside run() exactly: parabolic exit at 3.5xATR from the
    21 EMA, phase1 stop with a 200 EMA wick ratchet gated until 0.5R separation,
    breakeven at +1R, then a 21 EMA close trail in phase2. Used by the random
    entry control so the monkey runs through the identical exits."""
    R=abs(entry-stop)
    if R<=0: return None
    phase=1; st=float(stop)
    for t in range(t0+1,len(C)):
        ex=None
        if d>0:
            if H[t]-e21[t]>=3.5*A[t]: ex=e21[t]+3.5*A[t]
            if ex is None and phase==1:
                if L[t]<=st: ex=st
                elif H[t]>=entry+R: phase=2; st=entry
            if ex is None and phase==2:
                if L[t]<=entry: ex=entry
                elif C[t]<e21[t]: ex=C[t]
            if ex is None and phase==1 and e200[t]<C[t] and e200[t]>st and abs(e200[t]-entry)>=0.5*R: st=e200[t]
        else:
            if e21[t]-L[t]>=3.5*A[t]: ex=e21[t]-3.5*A[t]
            if ex is None and phase==1:
                if H[t]>=st: ex=st
                elif L[t]<=entry-R: phase=2; st=entry
            if ex is None and phase==2:
                if H[t]>=entry: ex=entry
                elif C[t]>e21[t]: ex=C[t]
            if ex is None and phase==1 and e200[t]>C[t] and e200[t]<st and abs(e200[t]-entry)>=0.5*R: st=e200[t]
        if ex is not None: return ((ex-entry)/R)*d
    return ((C[-1]-entry)/R)*d

def ema(x,n):
    a=2/(n+1); o=np.empty_like(x,float); o[0]=x[0]
    for i in range(1,len(x)): o[i]=a*x[i]+(1-a)*o[i-1]
    return o
def atr(h,l,c,n=14):
    tr=np.empty(len(c)); tr[0]=h[0]-l[0]
    for i in range(1,len(c)): tr[i]=max(h[i]-l[i],abs(h[i]-c[i-1]),abs(l[i]-c[i-1]))
    o=np.empty(len(c)); o[0]=tr[0]; a=1/n
    for i in range(1,len(c)): o[i]=a*tr[i]+(1-a)*o[i-1]
    return o
def pivots(h,l,k=3):
    n=len(h); ph=[];pl=[]
    for i in range(k,n-k):
        if h[i]==h[i-k:i+k+1].max(): ph.append(i)
        if l[i]==l[i-k:i+k+1].min(): pl.append(i)
    return ph,pl

def run(name,path,mult,start_ts,tol=0.0015,toltouch=0.0010,touch_band=0.0005,brk_tol=0.0002,SPANDAYS=7,K=3,MINSPAN=168,MINTOUCH=3,MAXSPAN=1200,CAP=4000.0,TGAP=6,causal=False,return_open=False,tick=0.0,slip_ticks=0.0):
    T,O,H,L,C=load_series(path); n=len(C)
    e21=ema(C,21);e200=ema(C,200);A=atr(H,L,C,14)
    PH,PL=pivots(H,L,K)
    start=int(np.searchsorted(T,start_ts))

    sh=[];rh=[];ih=0;il=0;last_ph=None;last_pl=None
    pos=None;trades=[];traded=set();FAN_S=[];FAN_R=[]
    def sl_low(i,j): return (L[j]-L[i])/(j-i)
    def sl_high(i,j): return (H[j]-H[i])/(j-i)
    def sepidx(idxs):
        out=[]
        for z in sorted(idxs):
            if not out or z-out[-1]>=TGAP: out.append(int(z))
        return out
    def refit(piv,arr,sign,a,m,t,hv):
        # fit a line whose FIRST anchor is a fan node (connectivity) and which spans 7+ calendar
        # days from anchor to entry, with 3+ reaching touches and no wick piercing it.
        cand=[]
        for p in piv:
            if p<a or p>t: continue
            if causal and p>t-K: continue   # only pivots confirmed as of bar t
            ln=arr[a]+m*(p-a); g=(arr[p]-ln) if sign>0 else (ln-arr[p])
            if -brk_tol*arr[p] <= g <= 0.005*arr[p]: cand.append(int(p))
        if len(cand)<MINTOUCH: return None
        best=None
        for x in range(len(cand)):
            p1=cand[x]
            if p1 not in hv: continue
            if (T[t]-T[p1]) < SPANDAYS*86400: continue
            for y in range(x+1,len(cand)):
                p2=cand[y]
                if p2-p1<TGAP: continue
                s=(arr[p2]-arr[p1])/(p2-p1)
                if (sign>0 and s<=0) or (sign<0 and s>=0): continue
                zz=np.arange(p1,t); lnz=arr[p1]+s*(zz-p1)
                if (sign>0 and np.any(L[p1:t]<lnz-brk_tol*L[p1:t])) or (sign<0 and np.any(H[p1:t]>lnz+brk_tol*H[p1:t])): continue
                lnt=arr[p1]+s*(t-p1); lnp=arr[p1]+s*(t-1-p1)
                if sign>0:
                    if not (L[t]<lnt-tol*L[t] and L[t-1]>=lnp-brk_tol*L[t-1]): continue
                else:
                    if not (H[t]>lnt+tol*H[t] and H[t-1]<=lnp+brk_tol*H[t-1]): continue
                if causal:
                    zz=np.arange(p1,t); lnz=arr[p1]+s*(zz-p1)
                    g=(arr[p1:t]-lnz) if sign>0 else (lnz-arr[p1:t])
                    mask=(g>=-brk_tol*arr[p1:t])&(g<=touch_band*arr[p1:t])
                    tt=sepidx([int(z) for z in zz[mask]])
                else:
                    tt=[]
                    for p in cand:
                        if p<p1 or p>t: continue
                        ln=arr[p1]+s*(p-p1); g=(arr[p]-ln) if sign>0 else (ln-arr[p])
                        if -brk_tol*arr[p] <= g <= touch_band*arr[p]: tt.append(p)
                    tt=sepidx(tt)
                if len(tt)>=MINTOUCH:
                    score=(len(tt),p2-p1)
                    if best is None or score>best[0]: best=(score,int(p1),float(s),int(p2),tt)
        if best is None: return None
        _,p1,s,p2,tt=best
        return (p1,s,p2,tt)

    for t in range(210,n):
        # confirm pivots into hulls (causal)
        while il<len(PL) and PL[il]+K<=t:
            p=PL[il]
            while len(sh)>=2 and sl_low(sh[-2],sh[-1])>=sl_low(sh[-1],p): sh.pop()
            if sh and sl_low(sh[-1],p)>0: FAN_S.append((int(sh[-1]),int(p)))
            sh.append(p); last_pl=p; il+=1
        while ih<len(PH) and PH[ih]+K<=t:
            p=PH[ih]
            while len(rh)>=2 and sl_high(rh[-2],rh[-1])<=sl_high(rh[-1],p): rh.pop()
            if rh and sl_high(rh[-1],p)<0: FAN_R.append((int(rh[-1]),int(p)))
            rh.append(p); last_ph=p; ih+=1

        # manage open position (exit engine: 200wick ratchet + BE@1R + 21close + parabolic)
        if pos:
            d=pos['dir'];ex=None;why=''
            if d>0:
                if H[t]-e21[t]>=3.5*A[t]: ex=e21[t]+3.5*A[t];why='parabolic'
                if ex is None and pos['phase']==1:
                    if L[t]<=pos['stop']: ex=pos['stop'];why='stop'
                    elif H[t]>=pos['entry']+pos['R']: pos['phase']=2;pos['stop']=pos['entry']
                if ex is None and pos['phase']==2:
                    if L[t]<=pos['entry']: ex=pos['entry'];why='BE'
                    elif C[t]<e21[t]: ex=C[t];why='21EMA'
            else:
                if e21[t]-L[t]>=3.5*A[t]: ex=e21[t]-3.5*A[t];why='parabolic'
                if ex is None and pos['phase']==1:
                    if H[t]>=pos['stop']: ex=pos['stop'];why='stop'
                    elif L[t]<=pos['entry']-pos['R']: pos['phase']=2;pos['stop']=pos['entry']
                if ex is None and pos['phase']==2:
                    if H[t]>=pos['entry']: ex=pos['entry'];why='BE'
                    elif C[t]>e21[t]: ex=C[t];why='21EMA'
            if ex is not None:
                rr=((ex-pos['entry'])/pos['R'])*d
                if slip_ticks and tick and pos['R']:        # entry + exit slippage, in R
                    rr-=(2.0*slip_ticks*tick)/pos['R']
                pos.update(exit_idx=int(t),exit=round(float(ex),4),R=round(rr,3),why=why,stop_final=round(float(pos['stop']),4),ratcheted=bool(abs(pos['stop']-pos['stop0'])>1e-6)); trades.append(pos); pos=None
            else:
                if pos['phase']==1:
                    if d>0 and e200[t]<C[t] and e200[t]>pos['stop'] and abs(e200[t]-pos['entry'])>=0.5*pos['R']: pos['stop']=e200[t]
                    if d<0 and e200[t]>C[t] and e200[t]<pos['stop'] and abs(e200[t]-pos['entry'])>=0.5*pos['R']: pos['stop']=e200[t]

        if pos is not None or t<start: continue

        # SUPPORT break -> SHORT : scan connected hull edges, oldest-shallowest first
        sig=None
        for i in range(1,len(sh)):
            a,b=sh[i-1],sh[i]
            if (a,b,'s') in traded: continue
            m=sl_low(a,b)
            if m<=0 or (t-a)<MINSPAN or (t-a)>MAXSPAN: continue
            lt=L[a]+m*(t-a); lp=L[a]+m*(t-1-a)
            if not (L[t]<lt-tol*L[t] and L[t-1]>=lp-tol*L[t-1]): continue
            # respected: no low below line over [a, t-1]
            zz=np.arange(a,t)
            if np.any(L[a:t] < L[a]+m*(zz-a) - tol*L[a:t]): continue
            rf=refit(PL,L,1,a,m,t,set(sh))
            if rf is None: continue
            a2,m2,last2,tch=rf
            lt=L[a2]+m2*(t-a2)
            floor=1.0*A[t]; entry=O[t] if O[t]<lt else lt
            cands=[]
            for j in range(1,len(rh)):
                aa,bb=rh[j-1],rh[j]; mm=(H[bb]-H[aa])/(bb-aa); val=H[aa]+mm*(t-aa)
                if val>entry+floor and (val-entry)*mult<=CAP and bb>=t-720 and aa>=t-2880: cands.append((float(val),int(aa),int(bb)))
            cands.sort()
            pick=cands[0] if cands else None
            if pick: stop=pick[0]; saf=dict(a=pick[1],ya=round(float(H[pick[1]]),4),b=pick[2],yb=round(float(H[pick[2]]),4),kind='res'); stop_src='auto'
            elif e200[t]>entry+floor and (e200[t]-entry)*mult<=CAP: stop=float(e200[t]); saf=None; stop_src='ema'
            else: continue
            Rd=stop-entry
            if Rd<=0: continue
            sig=dict(dir=-1,entry=round(float(entry),4),stop=round(float(stop),4),R=Rd,phase=1,t0=int(t),
                     a=int(a2),ya=round(float(L[a2]),4),m=float(m2),last=int(last2),kind='sup',tch=tch,stop0=round(float(stop),4),saf=saf,stop_src=stop_src); traded.add((a,b,'s')); break
        if sig is None:
            for i in range(1,len(rh)):
                a,b=rh[i-1],rh[i]
                if (a,b,'r') in traded: continue
                m=sl_high(a,b)
                if m>=0 or (t-a)<MINSPAN or (t-a)>MAXSPAN: continue
                lt=H[a]+m*(t-a); lp=H[a]+m*(t-1-a)
                if not (H[t]>lt+tol*H[t] and H[t-1]<=lp+tol*H[t-1]): continue
                zz=np.arange(a,t)
                if np.any(H[a:t] > H[a]+m*(zz-a) + tol*H[a:t]): continue
                rf=refit(PH,H,-1,a,m,t,set(rh))
                if rf is None: continue
                a2,m2,last2,tch=rf
                lt=H[a2]+m2*(t-a2)
                floor=1.0*A[t]; entry=O[t] if O[t]>lt else lt
                cands=[]
                for j in range(1,len(sh)):
                    aa,bb=sh[j-1],sh[j]; mm=(L[bb]-L[aa])/(bb-aa); val=L[aa]+mm*(t-aa)
                    if val<entry-floor and (entry-val)*mult<=CAP and bb>=t-720 and aa>=t-2880: cands.append((float(val),int(aa),int(bb)))
                cands.sort(reverse=True)
                pick=cands[0] if cands else None
                if pick: stop=pick[0]; saf=dict(a=pick[1],ya=round(float(L[pick[1]]),4),b=pick[2],yb=round(float(L[pick[2]]),4),kind='sup'); stop_src='auto'
                elif e200[t]<entry-floor and (entry-e200[t])*mult<=CAP: stop=float(e200[t]); saf=None; stop_src='ema'
                else: continue
                Rd=entry-stop
                if Rd<=0: continue
                sig=dict(dir=1,entry=round(float(entry),4),stop=round(float(stop),4),R=Rd,phase=1,t0=int(t),
                         a=int(a2),ya=round(float(H[a2]),4),m=float(m2),last=int(last2),kind='res',tch=tch,stop0=round(float(stop),4),saf=saf,stop_src=stop_src); traded.add((a,b,'r')); break
        if sig: pos=sig

    arr=np.array([x['R'] for x in trades]) if trades else np.array([])
    w=arr[arr>0.01];ls=arr[arr<-0.01]
    summ=dict(symbol=name,trades=len(trades),win=round(100*(arr>0.01).mean(),1) if arr.size else 0,
        totalR=round(arr.sum(),1) if arr.size else 0,avgR=round(arr.mean(),2) if arr.size else 0,
        PF=round(w.sum()/abs(ls.sum()),2) if ls.size else None,
        long=sum(1 for x in trades if x['dir']==1),short=sum(1 for x in trades if x['dir']==-1))
    if return_open:
        return summ,trades,FAN_S,FAN_R,pos   # pos is the position open at the last bar, or None
    return summ,trades,FAN_S,FAN_R


## 2. Config and the filename parser

In [ ]:
# CONFIG ---------------------------------------------------------------
import os, glob, re
import pandas as pd

DATA_DIR    = "futures_data"   # <-- put ALL your .txt / .csv files in this folder
MAXSPAN     = 1200             # validated best span (shorter structural lines win)
WARMUP_DAYS = 60               # history required before the first entry
SLIP_TICKS  = 0                # set 1 to net out 1 tick/side slippage (uses TICKS)
MIN_BARS    = 2000             # skip series too short to mean anything
MIN_TRADES  = 20               # only rank symbols with at least this many trades

# tick sizes only matter if SLIP_TICKS > 0; unknown symbols get no slippage
TICKS = {"MGC":0.10,"GC":0.10,"SI":0.005,"SIL":0.005,"HG":0.0005,"MHG":0.0005,
         "PL":0.10,"PA":0.05,"CL":0.01,"MCL":0.01,"NQ":0.25,"MNQ":0.25,
         "ES":0.25,"MES":0.25}

def symbol_from_filename(fn):
    """CL_full_1hour_continuous_ratio_adjusted.txt -> CL,  1782_MGC.csv -> MGC,  AD.csv -> AD"""
    stem = os.path.splitext(os.path.basename(fn))[0]
    if "_full" in stem:
        stem = stem.split("_full")[0]
    m = re.match(r"^\d+_(.+)$", stem)   # strip a leading "<digits>_" prefix
    if m:
        stem = m.group(1)
    return stem


## 3. Scan the whole folder
_Several minutes for 131 symbols._

In [ ]:
# SCAN EVERY FILE IN THE FOLDER ---------------------------------------
# Reads .txt AND .csv directly, no renaming needed. ~3-4s per symbol, so
# 131 symbols is roughly 8-10 minutes. Progress prints as it goes.
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")) +
               glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"found {len(files)} files in {DATA_DIR}/\n")

rows = []
for i, f in enumerate(files, 1):
    sym = symbol_from_filename(f)
    try:
        T = load_series(f)[0]
        if len(T) < MIN_BARS:
            print(f"[{i}/{len(files)}] {sym:6s} too short ({len(T)} bars), skip"); continue
        start_ts = int(T.min()) + WARMUP_DAYS * 86400
        s, tr, _, _ = run(sym, f, 1.0, start_ts, CAP=1e12, MAXSPAN=MAXSPAN,
                          tick=TICKS.get(sym, 0.0), slip_ticks=SLIP_TICKS)
        rows.append(dict(symbol=sym, trades=s["trades"], win=s["win"],
                         totalR=s["totalR"], avgR=s["avgR"], PF=s["PF"]))
        print(f"[{i}/{len(files)}] {sym:6s} {s['trades']:4d}tr  PF {s['PF']}  {s['totalR']:+.1f}R")
    except Exception as e:
        print(f"[{i}/{len(files)}] {sym:6s} error ({type(e).__name__}), skip")

df = pd.DataFrame(rows)
df = df[df["trades"] >= MIN_TRADES].sort_values("PF", ascending=False).reset_index(drop=True)
df.to_csv("scan_results.csv", index=False)
print(f"\n{len(df)} symbols with >= {MIN_TRADES} trades. saved -> scan_results.csv")


## 4. Ranked results

In [ ]:
# TOP RESULTS (ranked by profit factor) -------------------------------
# Anything PF > ~1.5 with a decent trade count is worth a closer look.
# Your known metals for reference: HG 2.26, PL 2.03, PA 1.98, MGC 1.85, SI 1.63.
df.head(30)


## 5. Batch monkey test: which symbols actually beat random
_Adds z_sigma and % of random that beat each symbol, then re-ranks. Anything that doesn't clear random is trend or overfit, not edge._

In [ ]:
# BATCH MONKEY TEST: flag which symbols actually beat random ----------
# Re-runs each symbol in the results table and shuffles entries+direction
# SIMS times. A high z and a low "% random beat" means the edge is in entry
# selection, not just trend. WARNING: this re-backtests every symbol, so it
# roughly doubles the scan time (another ~10-15 min for 100+ symbols).
SIMS = 300
fmap = {symbol_from_filename(f): f for f in files}
mrows = []
for i, sym in enumerate(df["symbol"], 1):
    f = fmap.get(sym)
    try:
        T, O, H, L, C = load_series(f)
        e21, e200, A = ema(C, 21), ema(C, 200), atr(H, L, C)
        st = int(T.min()) + WARMUP_DAYS * 86400
        s, tr, _, _ = run(sym, f, 1.0, st, CAP=1e12, MAXSPAN=MAXSPAN)
        if not tr:
            mrows.append((sym, np.nan, np.nan)); continue
        katr = float(np.median([abs(x["entry"] - x["stop0"]) / A[x["t0"]] for x in tr]))
        si = int(np.searchsorted(T, st)); n = len(tr)
        lo, hi = max(si, 210), len(C) - 60
        rng = np.random.default_rng(42); tot = np.zeros(SIMS)
        for it in range(SIMS):
            bars = rng.integers(lo, hi, size=n)
            dirs = rng.choice([-1, 1], size=n)
            tot[it] = sum(exit_sim(O, H, L, C, e21, e200, A, t0, O[t0], O[t0] - d * katr * A[t0], d)
                          for t0, d in zip(bars, dirs))
        z = (s["totalR"] - tot.mean()) / tot.std()
        beat = float((tot >= s["totalR"]).mean() * 100)
        mrows.append((sym, round(z, 1), beat))
        print(f"[{i}/{len(df)}] {sym:6s} z {z:5.1f}  {beat:4.1f}% of random beat")
    except Exception as e:
        print(f"[{i}/{len(df)}] {sym:6s} error ({type(e).__name__})")
        mrows.append((sym, np.nan, np.nan))

mk = pd.DataFrame(mrows, columns=["symbol", "z_sigma", "pct_random_beat"])
out = df.merge(mk, on="symbol").sort_values("z_sigma", ascending=False).reset_index(drop=True)
out["clears"] = (out["z_sigma"] >= 3) & (out["pct_random_beat"] <= 1.0)
out.to_csv("scan_results_monkey.csv", index=False)
print(f"\n{int(out['clears'].sum())} symbols clear the bar (z >= 3 and <= 1% of random beat). saved -> scan_results_monkey.csv")
out.head(40)


## 6. Per-period robustness: does the edge hold across regimes
_Splits each symbol's R by year. A real edge is spread across the sample with both halves positive. A regime result piles its R into one period. The final table is the shortlist that clears the monkey AND holds across periods._

In [ ]:
# PER-PERIOD ROBUSTNESS: does the edge hold across regimes, or one period? --
# Splits each symbol's trades by entry year. A real edge is spread across many
# years with BOTH halves of the sample positive. A regime or luck result piles
# most of its R into one period. This is the test win rate cannot give you, and
# it is the one that tells you whether a high PF survives a regime change.
# (Depends on `out` from the batch monkey cell. Run cell 5 first.)
from datetime import datetime, timezone
def _yr(ts): return datetime.fromtimestamp(ts, timezone.utc).year

prows = []
for i, sym in enumerate(df["symbol"], 1):
    f = fmap.get(sym)
    try:
        T = load_series(f)[0]
        st = int(T.min()) + WARMUP_DAYS * 86400
        s, tr, _, _ = run(sym, f, 1.0, st, CAP=1e12, MAXSPAN=MAXSPAN)
        if not tr:
            prows.append((sym, np.nan, np.nan, np.nan, "0/0", False)); continue
        by = {}
        for x in tr:
            y = _yr(T[x["t0"]]); by[y] = by.get(y, 0.0) + x["R"]
        tot = sum(by.values()); years = sorted(by)
        mid = years[0] + (years[-1] - years[0]) // 2
        h1 = sum(r for y, r in by.items() if y <= mid); h2 = tot - h1
        best_pct = 100 * max(by.values()) / tot if tot > 0 else 999
        npos = sum(1 for r in by.values() if r > 0); ny = len(years)
        robust = (h1 > 0) and (h2 > 0) and (best_pct <= 35) and (npos / ny >= 0.6)
        prows.append((sym, round(h1, 1), round(h2, 1), round(best_pct), f"{npos}/{ny}", robust))
        flag = "ROBUST" if robust else ("regime?" if best_pct > 45 or h1 <= 0 or h2 <= 0 else "")
        print(f"[{i}/{len(df)}] {sym:6s} H1 {h1:6.1f}  H2 {h2:6.1f}  best yr {best_pct:3.0f}%  +yrs {npos}/{ny}  {flag}")
    except Exception as e:
        print(f"[{i}/{len(df)}] {sym:6s} error ({type(e).__name__})")
        prows.append((sym, np.nan, np.nan, np.nan, "err", False))

pp = pd.DataFrame(prows, columns=["symbol", "H1_R", "H2_R", "best_yr_pct", "pos_years", "robust"])
final = out.merge(pp, on="symbol")
final.to_csv("scan_results_full.csv", index=False)
print(f"\n{int(final['robust'].sum())} symbols are period robust (both halves positive, no single year > 35%, positive in >= 60% of years). saved -> scan_results_full.csv")
# the real shortlist: clears the monkey AND holds across periods
final[final["clears"] & final["robust"]].sort_values("PF", ascending=False).head(40)


## 7. Optional: monkey-test a single symbol in detail

In [ ]:
# OPTIONAL: is one symbol's result luck? Random-entry monkey test -----
SYMBOL = "CL"          # <-- change to any symbol in your folder
SIMS   = 500

path = next((f for f in files if symbol_from_filename(f) == SYMBOL), None)
assert path, f"{SYMBOL} not found in {DATA_DIR}/"
T, O, H, L, C = load_series(path)
e21, e200, A = ema(C, 21), ema(C, 200), atr(H, L, C)
st = int(T.min()) + WARMUP_DAYS * 86400
s, tr, _, _ = run(SYMBOL, path, 1.0, st, CAP=1e12, MAXSPAN=MAXSPAN)
katr = float(np.median([abs(x["entry"] - x["stop0"]) / A[x["t0"]] for x in tr]))
si = int(np.searchsorted(T, st)); n = len(tr)
np.random.seed(42); tot = np.zeros(SIMS)
for it in range(SIMS):
    bars = np.random.randint(max(si, 210), len(C) - 60, size=n)
    dirs = np.random.choice([-1, 1], size=n)
    tot[it] = sum(exit_sim(O, H, L, C, e21, e200, A, t0, O[t0], O[t0] - d * katr * A[t0], d)
                  for t0, d in zip(bars, dirs))
z = (s["totalR"] - tot.mean()) / tot.std()
beat = (tot >= s["totalR"]).mean() * 100
print(f"{SYMBOL}: strategy {s['totalR']:+.1f}R vs random mean {tot.mean():+.1f}R "
      f"(std {tot.std():.1f}) -> {z:.1f} sigma, {beat:.1f}% of random beat it")
